# Phase 5 – Statistical Analysis & Hypothesis Testing

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.weightstats import ttest_ind
from statsmodels.stats.power import TTestIndPower
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("../Output/cleaned_data.csv")
df.head()

## Business Question
**H0:** Treatment conversion rate = Control conversion rate

**H1:** Treatment conversion rate > Control conversion rate

In [ ]:
control = df[df['group']=='Control']
treatment = df[df['group']=='Treatment']

conversion_summary = df.groupby('group')['converted'].agg(conversions='sum', users='count')

success = conversion_summary['conversions']
total = conversion_summary['users']

z_stat, conversion_p = proportions_ztest(count=success, nobs=total, alternative='larger')

print(z_stat, conversion_p)

In [ ]:
control_rate = control['converted'].mean()
treatment_rate = treatment['converted'].mean()

difference = treatment_rate - control_rate

se = np.sqrt((control_rate*(1-control_rate))/len(control)
             +(treatment_rate*(1-treatment_rate))/len(treatment))

lower = difference - 1.96*se
upper = difference + 1.96*se

print(lower, upper)

In [ ]:
t_stat, revenue_p, _ = ttest_ind(
    treatment['purchase_amount'],
    control['purchase_amount'],
    usevar='unequal')

session_stat, session_p, _ = ttest_ind(
    treatment['session_duration'],
    control['session_duration'],
    usevar='unequal')

pages_stat, pages_p, _ = ttest_ind(
    treatment['pages_visited'],
    control['pages_visited'],
    usevar='unequal')

In [ ]:
def cohens_d(g1,g2):
    pooled = np.sqrt((g1.var()+g2.var())/2)
    return (g1.mean()-g2.mean())/pooled

effect_size = cohens_d(
    treatment['session_duration'],
    control['session_duration'])

analysis = TTestIndPower()

power = analysis.power(effect_size=abs(effect_size),
                       nobs1=len(control),
                       alpha=0.05,
                       ratio=1)

print(effect_size, power)

In [ ]:
results=[]

results.append(['Conversion','Two-Proportion Z-Test',z_stat,conversion_p])
results.append(['Revenue','Welch T-Test',t_stat,revenue_p])
results.append(['Session Duration','Welch T-Test',session_stat,session_p])
results.append(['Pages Visited','Welch T-Test',pages_stat,pages_p])

summary = pd.DataFrame(results,
columns=['Metric','Test','Statistic','P-value'])

summary['Significant']=summary['P-value']<0.05

summary.to_csv('../Output/hypothesis_test_results.csv',index=False)

summary

## Interpretation
Reject H0 if p-value < 0.05. Report confidence intervals, effect size, statistical power, and provide a business recommendation.